# P1-32 · Paper 1: additional experiment W1 - faster-learning four-to-four mapping - session B (run only while session A is running)
**Runtime: T4 GPU.** Run all. Resumable: after a disconnect, disconnect and delete the runtime, reconnect and **Run all**.
QSEG_FINAL is read only; outputs (numbers only) go to `MyDrive/P1_v2/runs/W1_*`. The prespecified text is in cell 1 and is
written to `MyDrive/P1_v2/PROTOCOL_w1_prespecified.txt` with a time stamp before the first run. Cell 3b runs a pre-flight
check of the new layers and stops if it fails.

In [ ]:
# 1 · Settings, Drive, protocol record
SESSION = 'B'
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, shutil, random, gc
import numpy as np, pandas as pd
ROOT  = '/content/drive/MyDrive/QSEG_FINAL'        # read only
OUT   = '/content/drive/MyDrive/P1_v2'
EXT   = f'{OUT}/dmid_ext'
LOCAL = '/content/qseg_local'
TMP   = '/content/p1v2_tmp'
assert not OUT.startswith(ROOT)
os.makedirs(OUT + '/runs', exist_ok=True); os.makedirs(TMP, exist_ok=True)
assert os.path.exists(EXT + '/manifest.csv') and os.path.exists(OUT + '/mias_ext/manifest.csv'), 'Run notebooks P1-16b and P1-16c first'
PROTO = '''Paper 1 - additional experiment W1 (faster-learning four-to-four mapping), written before any of these runs was trained.

Motivation: with Adam at 1e-4 for every layer, the 20 (Dense, trigonometric) or 24 (circuit) weights of the four-to-four mapping can
move only about 1e-4 per step; over the observed training lengths no angle could move more than about 0.06-0.34 rad from its random
start, so the mapping may have stayed close to its initialization. W1 tests whether the circuit behaves differently when the
mapping learns faster.

Runs (15): E1 setting (hospital split, skip connections, Paper 1 architectures), HQB-UNet, HQB-cls and HQB-trig with a
    faster-learning mapping, seeds 0-4. Every weight of the mapping is stored as v = w / 100 and used as w = 100 v, so the mapping
    learns with an effective learning rate 100 times that of the rest of the network (1e-2 at the start, following the
    schedule); everything else is unchanged. The initial effective mapping weights equal those of the standard model of the
    same seed. All three models start from the initial weights of the HQB-UNet model of the same seed in every layer they share
    with it (checked before training; a run stops if the check fails).
Training and evaluation: exactly as in the v2 specification (PROTOCOL_prespecified.txt): Adam 1e-4, batch 16, BCE + soft Dice,
    up to 200 epochs, ReduceLROnPlateau on the training loss, early stopping on the validation loss (patience 10, from epoch 15),
    best validation weights tested; 84 test images (20 patients), external DMID and mini-MIAS sets; code replacement; finite-shot
    readout for HQB-UNet. Each run also records the change of the mapping weights and of the neighbouring layers from
    initialization to the tested weights.
Primary estimands: HQB-UNet - HQB-cls and HQB-UNet - HQB-trig (W1 models) in mean per-image test Dice, crossed hierarchical
    bootstrap over patients and seeds (10,000 replicates), Holm over these two; equivalence margin +-0.01 Dice as in E1.
Secondary (descriptive): each W1 model minus its standard E1 counterpart (same seeds); DMID and mini-MIAS contrasts;
    code-replacement drops; weight change.
All 15 runs are reported; none is discarded or repeated for its result; no further W1 runs are added.'''
proto = OUT + '/PROTOCOL_w1_prespecified.txt'
if not os.path.exists(proto):
    open(proto, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
print(open(proto).read().splitlines()[-1])


In [ ]:
# 2 · Data (local copy of the processed Paper 1 data; external DMID set)
if not os.path.exists(LOCAL + '/data/manifest.csv'):
    shutil.copytree(ROOT + '/results/data', LOCAL + '/data', ignore=shutil.ignore_patterns('*.npy', '*.h5'))
sys.path.insert(0, ROOT + '/code')
import tensorflow as tf
from tensorflow.keras import layers, models
assert tf.config.list_physical_devices('GPU'), 'No GPU: Runtime -> Change runtime type -> T4 (or L4) GPU'
from qseg.config import Config
from qseg.models import conv_block, output_layer, TFCircuitLayer
from qseg.datasets import load_manifest, make_dataset, load_arrays
from qseg.evaluation import per_image_overlap
from qseg.train import bce_dice_loss, soft_dice
cfg = Config(work_dir=LOCAL)
man = load_manifest(cfg)
X = {s: load_arrays(cfg, s, man) for s in ('train', 'val', 'test')}
em = pd.read_csv(EXT + '/manifest.csv'); em = em[em.duplicate_of.fillna('') == ''].reset_index(drop=True)
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
XE = np.stack([_png(f'{EXT}/images/{f}.png', False) for f in em.file_id])
YE = np.stack([_png(f'{EXT}/masks/{f}.png', True) for f in em.file_id])
MIAS = f'{OUT}/mias_ext'
mm = pd.read_csv(MIAS + '/manifest.csv'); mm = mm[mm.status == 'ok'].reset_index(drop=True)
XM = np.stack([_png(f'{MIAS}/images/{f}.png', False) for f in mm.file_id])
YM = {k: np.stack([_png(f'{MIAS}/{sub}/{f}.png', True) for f in mm.file_id])
      for k, sub in (('expert', 'masks_expert'), ('expert_breast', 'masks_expert_breast'), ('proxy', 'masks_proxy'))}
print({s: len(v[0]) for s, v in X.items()}, '| external DMID', XE.shape, '| mini-MIAS', XM.shape, '| TF', tf.__version__)


In [ ]:
# 3 · Models (Paper 1 hybrid architecture; optional removal of the skip connections) and evaluation helpers
class CodeSwitch(layers.Layer):
    """Identity during training; at test time can replace the four-value code by supplied codes."""
    def build(self, input_shape):
        self.on = self.add_weight(shape=(), initializer='zeros', trainable=False, name='on')
        self.buf = self.add_weight(shape=(cfg.batch_size, 4), initializer='zeros', trainable=False, name='buf')
    def call(self, z):
        n = tf.shape(z)[0]
        return z * (1.0 - self.on) + self.buf[:n] * self.on

class TrigLayer(layers.Layer):
    """20-parameter trigonometric 4 -> 4 map: z = tanh(A cos((pi/2) x + phi))."""
    def build(self, input_shape):
        self.A = self.add_weight(shape=(4, 4), initializer='glorot_uniform', name='A')
        self.phi = self.add_weight(shape=(4,), initializer=tf.keras.initializers.RandomUniform(0.0, 2 * np.pi), name='phi')
    def call(self, x):
        return tf.tanh(tf.matmul(tf.cos(np.pi / 2 * x + self.phi), self.A, transpose_b=True))

def build(kind, skips=True, filters=(32, 64, 128, 256, 512), dropouts=(0.1, 0.1, 0.2, 0.2, 0.3)):
    inp = layers.Input((256, 256, 1), name='input_image')
    sk, x = [], inp
    for i, (f, d) in enumerate(zip(filters[:-1], dropouts[:-1])):
        x = conv_block(x, f, d, name=f'enc{i + 1}'); sk.append(x)
        x = layers.MaxPooling2D(2, name=f'pool{i + 1}')(x)
    x = conv_block(x, filters[-1], dropouts[-1], name='bottleneck')
    z = layers.Dense(4, activation='tanh', name='pre_dense')(layers.GlobalAveragePooling2D(name='gap')(x))
    if kind == 'quantum':   z = TFCircuitLayer(4, 2, name='quantum_circuit')(z)
    elif kind == 'trig':    z = TrigLayer(name='trig_replacement')(z)
    elif kind == 'classical': z = layers.Dense(4, activation='tanh', name='classical_replacement')(z)
    z = CodeSwitch(name='code_switch')(z)
    q = layers.Dense(512, activation='relu', name='post_dense')(z)
    x = layers.Reshape((16, 16, filters[-1]), name='q_reshape')(
        layers.Dense(16 * 16 * filters[-1], activation='relu', name='q_expand')(q))
    for i, (f, s, d) in enumerate(zip(reversed(filters[:-1]), reversed(sk), reversed(dropouts[:-1]))):
        x = layers.UpSampling2D(2, name=f'up{i + 1}')(x)
        if skips: x = layers.Concatenate(name=f'concat{i + 1}')([x, s])
        x = conv_block(x, f, d, name=f'dec{i + 1}')
    return models.Model(inp, output_layer(x), name=f'HQB_{kind}_{"skip" if skips else "noskip"}')

def set_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def predict(model, x, codes=None):
    sw = model.get_layer('code_switch'); out = []
    for a in range(0, len(x), cfg.batch_size):
        xb = x[a:a + cfg.batch_size]
        if codes is None: sw.on.assign(0.0)
        else:
            buf = np.zeros((cfg.batch_size, 4), np.float32); buf[:len(xb)] = codes[a:a + len(xb)]
            sw.buf.assign(buf); sw.on.assign(1.0)
        out.append(model(xb, training=False).numpy()[..., 0])
    sw.on.assign(0.0)
    return np.concatenate(out)

def code_model(model):
    return tf.keras.Model(model.input, model.get_layer('code_switch').input)

def dice(Y, p):
    return per_image_overlap((Y[..., 0] > 0.5).astype(np.uint8), (p >= cfg.threshold).astype(np.uint8))[:, 0]

def shots(z, n, rng):
    """Finite-shot estimate of <Z> per qubit: p(+1) = (1 + z)/2, n shots."""
    p = np.clip((1 + z) / 2, 0, 1)
    return 2 * rng.binomial(n, p) / n - 1

def evaluate(model, kind, x, Y, z_tr):
    cm = code_model(model)
    z = cm.predict(x, batch_size=cfg.batch_size, verbose=0)
    d0 = dice(Y, predict(model, x))
    r = pd.DataFrame({'dice': d0})
    r['dice_check_switch'] = dice(Y, predict(model, x, z))             # must equal dice (sanity check)
    r['drop_const'] = d0 - dice(Y, predict(model, x, np.repeat(z_tr.mean(0)[None], len(x), 0)))
    for k in range(5):
        idx = np.random.default_rng(k).integers(0, len(z_tr), len(x))
        r[f'drop_swap{k}'] = d0 - dice(Y, predict(model, x, z_tr[idx]))
    r['drop_swapavg'] = r[[f'drop_swap{k}' for k in range(5)]].mean(1)
    if kind == 'quantum':
        for n in (100, 1000, 10000):
            ds = [dice(Y, predict(model, x, shots(z, n, np.random.default_rng(100 + k)))) for k in range(5)]
            r[f'dice_shots{n}'] = np.mean(ds, 0)
    r['code_sd'] = float(z.std(0).mean())
    return r


# ---- W1: faster-learning mapping (reparameterization w = SCALE * v) and shared initialization
SCALE = 100.0
SLOT = ('quantum_circuit', 'classical_replacement', 'trig_replacement')
class FastCircuit(TFCircuitLayer):
    def build(self, input_shape):
        super().build(input_shape); self.qweights.assign(self.qweights / SCALE)
    def call(self, inputs):
        from qseg.quantum_sim import circuit_expvals, tf_ops
        return circuit_expvals(tf.cast(inputs, tf.float32), SCALE * tf.cast(self.qweights, tf.float32), tf_ops(), self.n_qubits)
class FastDense(layers.Dense):
    def build(self, input_shape):
        super().build(input_shape); self.kernel.assign(self.kernel / SCALE); self.bias.assign(self.bias / SCALE)
    def call(self, x):
        return tf.tanh(tf.matmul(x, SCALE * self.kernel) + SCALE * self.bias)
class FastTrig(TrigLayer):
    def build(self, input_shape):
        super().build(input_shape); self.A.assign(self.A / SCALE); self.phi.assign(self.phi / SCALE)
    def call(self, x):
        return tf.tanh(tf.matmul(tf.cos(np.pi / 2 * x + SCALE * self.phi), SCALE * self.A, transpose_b=True))

_orig_classes = (TFCircuitLayer, TrigLayer)
def build_w1(kind, seed):
    """W1 model of `kind` with fast mapping; every layer shared with HQB-UNet copied from HQB-UNet of the same seed."""
    global TFCircuitLayer, TrigLayer
    set_seeds(seed); ref = build('quantum', True)                      # standard HQB-UNet, same seed
    refw = {l.name: l.get_weights() for l in ref.layers if l.weights}
    set_seeds(seed)
    if kind == 'quantum':
        TFCircuitLayer = FastCircuit
        try: m = build('quantum', True)
        finally: TFCircuitLayer = _orig_classes[0]
    elif kind == 'trig':
        TrigLayer = FastTrig
        try: m = build('trig', True)
        finally: TrigLayer = _orig_classes[1]
    else:
        m = build('classical_fast', True)
    copied = []
    for l in m.layers:
        if not l.weights or l.name not in refw: continue
        if l.name in SLOT:
            if kind == 'quantum': l.set_weights([w / SCALE for w in refw[l.name]])
            continue
        if [w.shape for w in l.get_weights()] == [w.shape for w in refw[l.name]]:
            l.set_weights(refw[l.name]); copied.append(l.name)
    same = all(np.array_equal(a, b) for l in m.layers if l.name in copied for a, b in zip(l.get_weights(), refw[l.name]))
    del ref; set_seeds(seed)
    return m, dict(copied=len(copied), shared_identical=bool(same and len(copied) > 10))

# HQB-cls with fast Dense: extend build() through a small switch on kind
_build_orig = build
def build(kind, skips=True, **kw):
    if kind != 'classical_fast': return _build_orig(kind, skips, **kw)
    import types
    g = dict(globals()); g['layers'] = types.SimpleNamespace(**{k: getattr(layers, k) for k in dir(layers) if not k.startswith('_')})
    g['layers'].Dense = lambda units, activation=None, name=None, **k2: (FastDense(units, activation=activation, name=name, **k2)
                                                                       if name == 'classical_replacement' else layers.Dense(units, activation=activation, name=name, **k2))
    f = types.FunctionType(_build_orig.__code__, g, 'build_fast_cls', _build_orig.__defaults__)
    return f('classical', skips, **kw)

def eff(layer):
    w = np.concatenate([v.numpy().astype(np.float64).ravel() for v in layer.trainable_weights])
    return w * SCALE if layer.name in SLOT else w
AUDIT = SLOT + ('pre_dense', 'post_dense')


In [ ]:
# 3b · Pre-flight check of the W1 models (about 1 min; stops if any check fails)
# eff() multiplies slot weights by SCALE (correct for W1 models); for the standard model the factor is undone here.
xb = X['train'][0][:2]; yb = X['train'][1][:2]
def one_step(m):
    opt = tf.keras.optimizers.Adam(cfg.learning_rate)
    with tf.GradientTape() as t: loss = bce_dice_loss(yb, m(xb, training=False))
    opt.apply_gradients(zip(t.gradient(loss, m.trainable_weights), m.trainable_weights))
for kind, slot in (('quantum', 'quantum_circuit'), ('classical', 'classical_replacement'), ('trig', 'trig_replacement')):
    tf.keras.backend.clear_session(); set_seeds(1); ms = build(kind, True); a0 = eff(ms.get_layer(slot)) / SCALE; os_ = ms(xb, training=False).numpy()
    one_step(ms); da = np.abs(eff(ms.get_layer(slot)) / SCALE - a0).max()
    tf.keras.backend.clear_session(); mf, rec = build_w1(kind, 1); b0 = eff(mf.get_layer(slot)); of = mf(xb, training=False).numpy()
    one_step(mf); db = np.abs(eff(mf.get_layer(slot)) - b0).max()
    out_diff = float(np.abs(os_ - of).max()) if kind == 'quantum' else float('nan')
    print(f'{kind:9s} init diff {np.abs(a0 - b0).max():.2e} | output diff (quantum only) {out_diff:.2e} | step ratio {db / max(da, 1e-12):.1f} | {rec}')
    assert np.abs(a0 - b0).max() < 1e-5 and 30 < db / max(da, 1e-12) < 1000 and rec['shared_identical'], 'pre-flight check failed'
    if kind == 'quantum': assert out_diff < 1e-4, 'pre-flight check failed (output)'
print('W1 PRE-FLIGHT CHECK PASSED')

In [ ]:
# 4 · Training and evaluation of all runs (resumable; two sessions share the work through lock files)
KIND = {'hq': 'quantum', 'hc': 'classical', 'hf': 'none', 'ht': 'trig'}
RUNS = [(f'W1_{m}', KIND[m], True, s) for s in range(5) for m in ('hq', 'hc', 'ht')]
assert len(RUNS) == 15
LOCK_H = 4
def claim(d):
    if os.path.exists(d + '/run.json'): return False
    lk = d + '/.lock'
    if os.path.exists(lk):
        who, t = open(lk).read().split()
        if who != SESSION and time.time() - float(t) < LOCK_H * 3600: return False
    os.makedirs(d, exist_ok=True); open(lk, 'w').write(f'{SESSION} {time.time()}')
    time.sleep(20)
    return open(lk).read().split()[0] == SESSION

def run(name, kind, skips, seed):
    d = f'{OUT}/runs/{name}_seed{seed}'
    if not claim(d): return
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.mixed_precision.set_global_policy('float32'); set_seeds(seed)
    tr, _ = make_dataset(cfg, 'train', seed, man); va, _ = make_dataset(cfg, 'val', seed, man)
    model, init_rec = build_w1(kind, seed)
    assert init_rec['shared_identical'], 'shared initialization check failed'
    w0 = {l.name: eff(l) for l in model.layers if l.name in AUDIT}
    model.compile(optimizer=tf.keras.optimizers.Adam(cfg.learning_rate), loss=bce_dice_loss, metrics=[soft_dice], jit_compile=False)
    ck = f'{TMP}/{name}_{seed}.weights.h5'
    cbs = [tf.keras.callbacks.TerminateOnNaN(),
           tf.keras.callbacks.ModelCheckpoint(ck, monitor='val_loss', save_best_only=True, save_weights_only=True),
           tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, start_from_epoch=15),
           tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=4, min_lr=1e-7),
           tf.keras.callbacks.CSVLogger(d + '/history.csv')]
    t0 = time.time()
    h = model.fit(tr, validation_data=va, epochs=200, callbacks=cbs, verbose=0)
    secs = time.time() - t0
    model.load_weights(ck)
    wc = {}
    for l in model.layers:
        if l.name in w0:
            dd = eff(l) - w0[l.name]
            if l.name == 'quantum_circuit': dd = (dd + np.pi) % (2 * np.pi) - np.pi
            wc[l.name] = dict(n=int(dd.size), max_abs_change=float(np.abs(dd).max()), rms_change=float(np.sqrt((dd ** 2).mean())),
                              rel_change=float(np.linalg.norm(dd) / max(np.linalg.norm(w0[l.name]), 1e-12)))
    json.dump(dict(init=init_rec, weight_change=wc, slot_scale=SCALE), open(d + '/weight_change.json', 'w'), indent=1)
    z_tr = code_model(model).predict(X['train'][0], batch_size=cfg.batch_size, verbose=0)
    xt, yt, dft = X['test']
    te = evaluate(model, kind, xt, yt, z_tr); te.insert(0, 'file_id', dft.file_id.values); te.insert(1, 'patient_id', dft.patient_id.values)
    ex = evaluate(model, kind, XE, YE, z_tr); ex.insert(0, 'file_id', em.file_id.values)
    pm = predict(model, XM)
    mi = pd.DataFrame({'file_id': mm.file_id.values, 'dice_expert': dice(YM['expert'], pm),
                       'dice_expert_breast': dice(YM['expert_breast'], pm), 'dice_proxy': dice(YM['proxy'], pm)})
    mi.to_csv(d + '/mias_per_image.csv', index=False)
    np.savez_compressed(d + '/test_prob_uint8.npz', prob=(predict(model, xt) * 255).round().astype(np.uint8), file_id=dft.file_id.values)
    np.savez_compressed(d + '/dmid_prob_uint8.npz', prob=(predict(model, XE) * 255).round().astype(np.uint8), file_id=em.file_id.values)
    np.savez_compressed(d + '/mias_prob_uint8.npz', prob=(pm * 255).round().astype(np.uint8), file_id=mm.file_id.values)
    va_d = dice(X['val'][1], predict(model, X['val'][0]))
    te.to_csv(d + '/test_per_image.csv', index=False); ex.to_csv(d + '/dmid_per_image.csv', index=False)
    hist = pd.DataFrame(h.history); vl = hist['val_loss'].to_numpy()
    lr = hist['learning_rate'] if 'learning_rate' in hist else hist.get('lr', pd.Series([np.nan]))
    r = dict(name=name, kind=kind, skips=skips, seed=seed, epochs_run=len(vl), best_epoch=int(np.nanargmin(vl)) + 1,
             final_lr=float(lr.iloc[-1]), minutes=round(secs / 60, 1), params=int(model.count_params()),
             val_dice=float(va_d.mean()), test_dice=float(te.dice.mean()), dmid_dice=float(ex.dice.mean()),
             mias_dice_expert=float(mi.dice_expert.mean()), mias_dice_proxy=float(mi.dice_proxy.mean()),
             test_drop_const=float(te.drop_const.mean()), test_drop_swapavg=float(te.drop_swapavg.mean()),
             switch_check_max=float(np.abs(te.dice - te.dice_check_switch).max()), session=SESSION,
             slot_max_change=max(v['max_abs_change'] for k, v in wc.items() if k in SLOT))
    json.dump(r, open(d + '/run.json', 'w'), indent=2)
    os.remove(ck); del model; tf.keras.backend.clear_session(); gc.collect()
    done = sum(os.path.exists(f'{OUT}/runs/{a}_seed{s}/run.json') for a, _, _, s in RUNS)
    print(time.strftime('%H:%M UTC', time.gmtime()), f"| {name} seed {seed}: {r['epochs_run']} ep (best {r['best_epoch']}), "
          f"{r['minutes']} min | test Dice {r['test_dice']:.4f} | DMID {r['dmid_dice']:.4f} | MIAS expert {r['mias_dice_expert']:.4f} | drop swap {r['test_drop_swapavg']:.4f} "
          f"| slot max change {r['slot_max_change']:.3f} | finished {done} / {len(RUNS)}", flush=True)

order = RUNS if SESSION == 'A' else RUNS[::-1]
for args in order:
    run(*args)
left = [a for a in RUNS if not os.path.exists(f'{OUT}/runs/{a[0]}_seed{a[3]}/run.json')]
print(f'nothing left for this session; {len(left)} run(s) still running in the other session')


In [ ]:
# 5 · Overview of finished runs
rows = [json.load(open(f'{OUT}/runs/{a}_seed{s}/run.json')) for a, _, _, s in RUNS if os.path.exists(f'{OUT}/runs/{a}_seed{s}/run.json')]
ov = pd.DataFrame(rows); ov.to_csv(OUT + '/overview_w1.csv', index=False)
print(ov.groupby('name')[['test_dice', 'dmid_dice', 'mias_dice_expert', 'test_drop_const', 'test_drop_swapavg', 'epochs_run', 'best_epoch', 'slot_max_change']].mean().round(4).to_string())
print(ov.groupby('name').size().to_string())


In [ ]:
# 6 · Auto-shutdown
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()
